# E-Ticaret Oturum Analizi
### Ziyaretçi Davranışlarından Satın Alma Tahmini

**Hazırlayan:** Damla Karaman

## Problem ve amaç
Bir e-ticaret sitesindeki ziyaretçinin gezinme davranışlarını kullanarak,
oturumun satın almayla sonuçlanıp sonuçlanmadığını tahmin etmek.

Bu çalışma, satın almayla ilişkili davranışları inceleyen ve geçmiş
oturum özetleri üzerinde çalışan bir sınıflandırma prototipidir.

## Hedef değişken
Revenue:
- False: Satın alma gerçekleşmedi.
- True: Satın alma gerçekleşti.

## Yöntem
Önce çoğunluk sınıfını tahmin eden bir referans yöntem
(DummyClassifier), ardından bir karar ağacı (DecisionTreeClassifier)
kullanılacaktır.

## Veri kaynağı
Online Shoppers Purchasing Intention Dataset — UCI.
Kaggle: https://www.kaggle.com/datasets/imakash3011/online-shoppers-purchasing-intention-dataset

## Proje kararları ve rapor notları
- Revenue hedef değişkendir; model girdilerine dahil edilmeyecektir.
- PageValues, hesaplanmasında işlem değerleriyle ilişkili bilgi
  kullanıldığı için ilk modelden çıkarılacaktır. Bu karar, doğrulanmış
  veri sızıntısı bulgusu olarak sunulmayacaktır.
- Sınıf dağılımı incelenecek; accuracy yanında precision, recall ve F1
  raporlanacaktır.
- Ön işleme yalnızca eğitim verisinden öğrenilecektir.
- Model ayarları eğitim verisi içinde seçilecek; test verisi son
  değerlendirme için ayrılacaktır.
- Tamamlanmış oturum özetleriyle çalışıldığı için gerçek zamanlı
  tahmin performansı iddia edilmeyecektir.
- Model sonuçları, indirim veya kampanyanın nedensel etkisini göstermez.
- Bulgular ve başarı değerleri kod çıktıları alındıktan sonra yazılacaktır.

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
from pathlib import Path
import hashlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Drive'daki klasör adını ekran görüntüsüne göre kullanıyoruz.
PROJE_KLASORU = Path("/content/drive/MyDrive/BasitlestirilmisPy")
VERI_DOSYASI = PROJE_KLASORU / "online_shoppers_intention.csv"
CIKTI_KLASORU = PROJE_KLASORU / "ciktilar"

if not VERI_DOSYASI.is_file():
    raise FileNotFoundError(
        f"CSV bulunamadı: {VERI_DOSYASI}\n"
        "Drive bağlantısını, klasör adını ve CSV dosya adını kontrol et."
    )

CIKTI_KLASORU.mkdir(parents=True, exist_ok=True)

print("CSV bulundu:", VERI_DOSYASI.name)
print("Çıktı klasörü hazır:", CIKTI_KLASORU)
print("Pandas sürümü:", pd.__version__)

CSV bulundu: online_shoppers_intention.csv
Çıktı klasörü hazır: /content/drive/MyDrive/BasitlestirilmisPy/ciktilar
Pandas sürümü: 2.2.3


In [6]:
# Ham veriyi koruyor, inceleme için ayrı bir kopya oluşturuyoruz.
df_ham = pd.read_csv(VERI_DOSYASI)
df = df_ham.copy()

# Aynı veri dosyasını kullandığımızı kontrol edebilmek için parmak izi.
veri_sha256 = hashlib.sha256(VERI_DOSYASI.read_bytes()).hexdigest()

print(f"Satır sayısı: {df.shape[0]:,}")
print(f"Sütun sayısı: {df.shape[1]}")
print("Veri dosyası SHA-256:", veri_sha256)

display(df.head())

# Veri kaynağı bilgilerini Drive'a kaydet.
kaynak_notu = (
    "Veri seti: Online Shoppers Purchasing Intention Dataset\n"
    "Asıl kaynak: UCI Machine Learning Repository\n"
    "Kaggle: https://www.kaggle.com/datasets/imakash3011/"
    "online-shoppers-purchasing-intention-dataset\n"
    f"Dosya: {VERI_DOSYASI.name}\n"
    f"Boyut: {df.shape[0]} satır, {df.shape[1]} sütun\n"
    f"SHA-256: {veri_sha256}\n"
)

(CIKTI_KLASORU / "veri_kaynagi.txt").write_text(
    kaynak_notu, encoding="utf-8"
)

Satır sayısı: 12,330
Sütun sayısı: 18
Veri dosyası SHA-256: b3055ee355f59134d851d32641183cb4a8b45def7124d2f50442a042f358e0d9


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


338

In [7]:
if "Revenue" not in df.columns:
    raise ValueError("Revenue sütunu bulunamadı.")

if df["Revenue"].isna().any():
    raise ValueError("Hedef sütununda eksik değer var; incelemeliyiz.")

print("Revenue veri tipi:", df["Revenue"].dtype)
print("Revenue değerleri:", df["Revenue"].unique())

hedef_ozeti = df["Revenue"].value_counts(dropna=False).to_frame(
    name="oturum_sayisi"
)
hedef_ozeti["oran_yuzde"] = (
    hedef_ozeti["oturum_sayisi"] / len(df) * 100
).round(2)

display(hedef_ozeti)

hedef_ozeti.to_csv(
    CIKTI_KLASORU / "hedef_dagilimi.csv",
    encoding="utf-8-sig",
    index_label="Revenue"
)

Revenue veri tipi: bool
Revenue değerleri: [False  True]


,oturum_sayisi,oran_yuzde
Revenue,,
False,10422,84.53
True,1908,15.47
